In [ ]:
%reset -f
import nlopt
import mfem.ser as mfem
import numpy as np

#### Problema de superficie mínima
Se trata de encontrar la función $u:\Omega \longrightarrow \mathbb{R}$ que minimiza
$$\int_\Omega \sqrt{1 + |\nabla u|^2}$$
entre todas las funciones $u\in u_0 + H^1_0(\Omega)$, con $u_0$ dada.

Consideramos el funcional 
$$J(u) = \int_\Omega \sqrt{1 + |\nabla u|^2}$$
cuya derivada en la dirección $v$ es:
$$\delta J(u; v) = \int_\Omega \frac{1}{\sqrt{1+ |\nabla u|^2}}\nabla u\nabla v$$
para $v\in H^1_0(\Omega)$.

#### Coeficientes para implementar los funcionales

In [ ]:
class Area(mfem.PyCoefficientBase):
    def __init__(self, u):
        super(Area, self).__init__(0)
        self.u = u 
        self.grad = mfem.Vector()
    def Eval(self, T, ip):
        self.u.GetGradient(T, self.grad)
        sig = 1 + self.grad*self.grad
        return np.sqrt(sig)

class InverArea(mfem.VectorPyCoefficientBase):
    def __init__(self, dim, u):
        super(InverArea, self).__init__(dim, 0)
        self.u = u 
    def Eval(self, elvect, T, ip):
        self.u.GetGradient(T, elvect)
        sig = 1 + elvect*elvect
        elvect *= 1./np.sqrt(sig)

In [ ]:
class Initial(mfem.PyCoefficient):
    def EvalValue(self,x):
        return np.cos(np.sin(x[0]*x[1]))
        
class MyBoundary(mfem.PyCoefficient):
    def EvalValue(self,x):
        return 0.        

#### Resolución mediante NLOPT
Dada una función $f$, se define una función que calcula $f(x)$ y $\nabla f(x)$, y se le pasa a NLOPT del siguiente modo:
<code>
opt = nlopt.opt(nlopt.LD_LBFGS, n) # seleccionamos optimizador
opt.set_min_objective(myfunc) # myfunc calcula f y grad f
opt.set_xtol_rel(1.e-4) # parámetros de optimización
opt.set_maxeval(200) # 
opt.optimize(x) # optimización
</code>

Definimos una clase que permite calcular el valor de la función objetivo

In [ ]:
class Surface:
    def __init__(self, u_, ess_, sout_):
        self.mu_gf = u_
        self.ess = ess_
        self.sout = sout_
        fespace = u_.FESpace()
        self.iter = 0
        self.mesh = fespace.GetMesh()
        dim = self.mesh.Dimension()
        self.vol = mfem.LinearForm(fespace)
        pp_coeff = Area(self.mu_gf)
        self.vol.AddDomainIntegrator(mfem.DomainLFIntegrator(pp_coeff))
        ipp_coeff = InverArea(dim, self.mu_gf)
        self.a = mfem.LinearForm(fespace)
        self.a.AddDomainIntegrator(mfem.DomainLFGradIntegrator(ipp_coeff))
        self.X = mfem.GridFunction(fespace)
        
    def myfunc(self, x, grad):
        self.mu_gf.SetVector(mfem.Vector(x),0)
        self.vol.Assemble()
        C = self.vol.Sum()
        
        if grad.size > 0:
            self.a.Assemble()
            self.X.SetVector(self.a,0)
            self.X.ProjectBdrCoefficient(mfem.ConstantCoefficient(0.),self.ess)
            grad[:] = self.X.GetDataArray()
            
        self.iter += 1
        print("Iter:",self.iter,"Valor:",C)
        if not self.iter % 2:
            self.sout << "solution\n" << self.mesh << self.mu_gf
            
        return C


#### Definición de malla y espacio de elementos finitos

In [ ]:
#  malla
mesh = mfem.Mesh.MakeCartesian2D(50,50,mfem.Geometry.SQUARE,sx = np.pi, sy = np.pi)
#mesh = mfem.Mesh("minicuadrado.mesh")
ess = mfem.intArray([1,1,1,1])

In [ ]:
# espacio de elementos finitos
fec = mfem.H1_FECollection(1,  mesh.Dimension())
fespace = mfem.FiniteElementSpace(mesh, fec)

u = mfem.GridFunction(fespace)
n = fespace.GetTrueVSize()

Función inicial, con valores en la frontera fijados

In [ ]:
coeff = Initial()
boundary = MyBoundary()
u.ProjectCoefficient(coeff)
u.ProjectBdrCoefficient(boundary,ess)
x = u.GetDataArray()

In [ ]:
sout = mfem.socketstream("localhost", 19916)
sout.precision(8)
sout << "solution\n" << mesh << u
sout.send_text("valuerange 0 1\n")
sout.send_text("autoscale off\n")
sout.send_text("keys c\n")

#### Creación del objeto NLOPT

In [ ]:
obj = Surface(u, ess, sout)
opt = nlopt.opt(nlopt.LD_LBFGS, n)
opt.set_min_objective(obj.myfunc)
opt.set_xtol_rel(1.e-4)
opt.set_maxeval(100)

##### Optimización y resultados

In [ ]:
x0 = opt.optimize(x)

opt.last_optimum_value()
u.SetVector(mfem.Vector(x0),0)
sout << "solution\n" << mesh << u